In [1]:
import onnx
import os
from collections import Counter

from pathlib import Path
from hailo_sdk_client import ClientRunner

[warning] Cannot use graphviz, so no visualizations will be created
[info] No GPU chosen and no suitable GPU found, falling back to CPU.


E0000 00:00:1795316614.783379   10622 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795316614.813567   10622 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795316614.982611   10622 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795316614.982683   10622 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795316614.982692   10622 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795316614.982698   10622 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
ONNX_PATH = Path("~/models/snowflake-arctic-m-v2/onnx/model_fp16.onnx").expanduser()
MODEL_NAME = "snowflake_arctic_embed_m_v2_full"
HAR_PATH = Path(f"{MODEL_NAME}.har")
assert ONNX_PATH.exists(), f"Missing ONNX: {ONNX_PATH}"
model = onnx.load(ONNX_PATH, load_external_data=False)

In [3]:
def info(v):
    t=v.type.tensor_type
    dims=[d.dim_value if d.HasField("dim_value") else d.dim_param for d in t.shape.dim]
    return v.name,onnx.TensorProto.DataType.Name(t.elem_type),dims

In [4]:
print("Inputs:")
for x in model.graph.input:
    print(x.name)

print("\nOutputs:")
for x in model.graph.output:
    print(x.name)

print("\nOperators:")
print(Counter(n.op_type for n in model.graph.node))

Inputs:
input_ids
attention_mask

Outputs:
token_embeddings
sentence_embedding

Operators:
Counter({'Add': 159, 'Mul': 135, 'Gather': 103, 'Unsqueeze': 101, 'Concat': 73, 'MatMul': 72, 'Slice': 51, 'Shape': 50, 'ReduceMean': 50, 'Reshape': 48, 'Transpose': 48, 'Div': 38, 'Sub': 26, 'Pow': 25, 'Sqrt': 25, 'Split': 24, 'Neg': 24, 'Softmax': 12, 'Erf': 12, 'Cast': 3, 'Expand': 2, 'Equal': 1, 'Where': 1, 'ReduceL2': 1, 'Clip': 1})


In [5]:
print("INPUTS/OUTPUTS:")
for v in list(model.graph.input)+list(model.graph.output): print(info(v))
print("\nINPUT CONSUMERS:")
for key in ("input_ids","attention_mask"):
    for n in model.graph.node:
        if key in n.input: print(key,n.op_type,n.name,list(n.output)[:2])
print("\nOUTPUT PRODUCERS:")
for n in model.graph.node:
    if "token_embeddings" in n.output or "sentence_embedding" in n.output:
        print(n.op_type,n.name,list(n.input)[:3])
print("\nFIRST 8 GATHER NODES:")
for n in [n for n in model.graph.node if n.op_type=="Gather"][:8]:
    print(n.name,list(n.input)[:2],list(n.output)[:1])

INPUTS/OUTPUTS:
('input_ids', 'INT64', ['batch_size', 'sequence_length'])
('attention_mask', 'INT64', ['batch_size', 'sequence_length'])
('token_embeddings', 'FLOAT', ['batch_size', 'sequence_length', 768])
('sentence_embedding', 'FLOAT', ['batch_size', 768])

INPUT CONSUMERS:
input_ids Shape /0/auto_model/embeddings/Shape ['/0/auto_model/embeddings/Shape_output_0']
input_ids Gather /0/auto_model/embeddings/word_embeddings/Gather ['/0/auto_model/embeddings/word_embeddings/Gather_output_0']
attention_mask Unsqueeze /0/auto_model/Unsqueeze ['/0/auto_model/Unsqueeze_1_output_0']

OUTPUT PRODUCERS:
Cast graph_output_cast0 ['graph_output_cast_0']
Cast graph_output_cast1 ['graph_output_cast_1']

FIRST 8 GATHER NODES:
/0/auto_model/embeddings/word_embeddings/Gather ['0.auto_model.embeddings.word_embeddings.weight', 'input_ids'] ['/0/auto_model/embeddings/word_embeddings/Gather_output_0']
/0/auto_model/embeddings/Gather ['/0/auto_model/embeddings/Shape_output_0', '/0/auto_model/embeddings/Cons

In [ ]:
print(f"Loading: {ONNX_PATH}")
print(f"Model size: {ONNX_PATH.stat().st_size / 1e6:.1f} MB")
print("Target: Hailo-10H")
print("Input: batch=1, sequence=128")

runner = ClientRunner(hw_arch="hailo10h")

print("\nStarting ONNX translation...")

hn, npz = runner.translate_onnx_model(
    str(ONNX_PATH),
    MODEL_NAME,
    net_input_shapes={"input_ids": [1, 128], "attention_mask": [1, 128]})

print("\nONNX translation successful!")

runner.save_har(str(HAR_PATH))

print(f"HAR saved: {HAR_PATH.resolve()}")
print(f"HAR size: {HAR_PATH.stat().st_size / 1e6:.2f} MB")

Loading: /home/totmalone-dell-campus/models/snowflake-arctic-m-v2/onnx/model_fp16.onnx
Model size: 613.3 MB
Target: Hailo-10H
Input: batch=1, sequence=128

Starting ONNX translation...
[info] Translation started on ONNX model snowflake_arctic_embed_m_v2_full
[info] Restored ONNX model snowflake_arctic_embed_m_v2_full (completion time: 00:00:06.16)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:47.16)
[warning] ONNX shape inference failed: [ONNXRuntimeError] : 10 : INVALID_GRAPH : This is an invalid model. Type Error: Type 'tensor(float)' of input parameter (input_ids) of operator (Gather) in node (/0/auto_model/embeddings/word_embeddings/Gather) is invalid.
